# Price Prediction for Agricultural Commodities

This notebook demonstrates how to predict agricultural commodity prices using time series data and machine learning models. We will walk through data loading, preprocessing, feature engineering, model training (Linear Regression and Random Forest), and evaluation.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


## 1. Library Imports

This cell imports necessary Python libraries:
*   `pandas` for data manipulation and analysis.
*   `numpy` for numerical operations, especially for array handling.
*   `warnings` to manage warnings, specifically to filter out any minor warnings that might clutter the output.

In [2]:
from google.colab import files
import io

print("Upload Karnataka_Processed.csv")
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(df.shape)
df.head()

Upload Karnataka_Processed.csv


Saving Karnataka_Processed.csv to Karnataka_Processed.csv
(12774, 10)


,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
0,Karnataka,bangalore,Ramanagara,Tomato,Tomato,FAQ,2000,2800,2500,6/06/2023
1,Karnataka,bangalore,Doddaballa Pur,Onion,Onion,FAQ,1400,2000,1700,6/06/2023
2,Karnataka,bangalore,Bangalore,Onion,Bangalore-Samall,FAQ,300,600,500,6/06/2023
3,Karnataka,bangalore,Doddaballa Pur,Tomato,Tomato,FAQ,1000,1600,1300,6/06/2023
4,Karnataka,bellary,Hospet,Tomato,Tomato,FAQ,700,1000,800,6/06/2023


## 2. Data Loading

This cell handles the upload and loading of the `Karnataka_Processed.csv` dataset. It uses `files.upload()` for interactive file upload in Colab, reads the CSV into a pandas DataFrame, and then displays its shape and the first few rows to give an initial overview of the data.

In [5]:
df['Price Date'] = pd.to_datetime(df['Price Date'], format='%d/%m/%Y')

onion_df = df[df['Commodity'] == 'Onion'].copy()

onion_df = onion_df.groupby('Price Date')['Modal_Price'].mean().reset_index()

onion_df = onion_df.sort_values('Price Date')

print(onion_df.head())
print(f"Total Days: {len(onion_df)}")

  Price Date  Modal_Price
0 2023-06-06      1154.00
1 2023-06-07      1149.75
2 2023-06-08      1204.05
3 2023-06-09      1224.00
4 2023-06-10      1103.00
Total Days: 586


## 3. Data Preprocessing for Onion Prices

This section preprocesses the data specifically for 'Onion' commodity prices:

1.  **Convert 'Price Date'**: The 'Price Date' column is converted to datetime objects, which is crucial for time-series analysis.
2.  **Filter for 'Onion'**: The DataFrame is filtered to include only entries where 'Commodity' is 'Onion'. A `.copy()` is used to avoid `SettingWithCopyWarning`.
3.  **Group and Aggregate**: Data is grouped by 'Price Date', and the `Modal_Price` is averaged for each day to get a single daily price.
4.  **Sort by Date**: The DataFrame is sorted by 'Price Date' to ensure chronological order, which is essential for creating time-series features.
5.  **Display Info**: The head of the processed `onion_df` and the total number of unique days are printed.

In [6]:
LOOKBACK = 30

prices = onion_df['Modal_Price'].values

X = []
y = []

for i in range(LOOKBACK, len(prices)):
    X.append(prices[i-LOOKBACK:i])
    y.append(prices[i])

X = np.array(X)
y = np.array(y)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (556, 30)
y Shape: (556,)


## 4. Feature Engineering: Creating Lagged Features

This cell prepares the dataset for time-series prediction by creating lagged features:

*   **`LOOKBACK`**: Defines the number of previous days' prices to use as features for predicting the current day's price. Here, we use the last 30 days.
*   **`prices`**: Extracts the 'Modal_Price' values into a NumPy array.
*   **`X` (Features)**: For each day, `X` stores the `LOOKBACK` number of previous days' prices.
*   **`y` (Target)**: For each day, `y` stores the current day's price, which is what we aim to predict.

The `X` and `y` arrays are then converted to NumPy arrays, and their shapes are printed to verify the data structure.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (444, 30)
Test : (112, 30)


## 5. Splitting Data into Training and Testing Sets

This cell splits the prepared `X` and `y` data into training and testing sets using `sklearn.model_selection.train_test_split`.

*   **`test_size=0.2`**: 20% of the data will be used for testing, and 80% for training.
*   **`shuffle=False`**: This is crucial for time-series data to maintain the chronological order. Shuffling would mix future data into the training set, leading to data leakage.

The shapes of the training and testing sets for both features (`X`) and target (`y`) are printed to confirm the split.

In [8]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("✅ Linear Regression trained!")

✅ Linear Regression trained!


## 6. Training Linear Regression Model (Onion Prices)

This cell initializes and trains a `LinearRegression` model from `sklearn.linear_model`.

*   A `LinearRegression` model is instantiated.
*   The model is then fitted to the training data (`X_train`, `y_train`). This step learns the linear relationship between the lagged prices (features) and the current day's price (target).

A confirmation message is printed once the model is successfully trained.

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = lr_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

RMSE : 350.38
MAE  : 223.19
R²   : 0.5923


## 7. Evaluating Linear Regression Model (Onion Prices)

This cell evaluates the performance of the trained Linear Regression model on the test set for 'Onion' prices using several regression metrics:

1.  **Predictions**: The `lr_model.predict(X_test)` method generates price predictions for the test features.
2.  **Root Mean Squared Error (RMSE)**: Calculated using `np.sqrt(mean_squared_error(y_test, y_pred))`. It measures the average magnitude of the errors, giving higher weight to larger errors.
3.  **Mean Absolute Error (MAE)**: Calculated using `mean_absolute_error(y_test, y_pred)`. It measures the average absolute difference between predicted and actual values.
4.  **R-squared (R²)**: Calculated using `r2_score(y_test, y_pred)`. It indicates the proportion of the variance in the dependent variable that is predictable from the independent variables.

These metrics help assess how well the Linear Regression model performs in predicting 'Onion' prices.

In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("✅ Random Forest trained!")

✅ Random Forest trained!


## 8. Training Random Forest Regressor Model (Onion Prices)

This cell initializes and trains a `RandomForestRegressor` model from `sklearn.ensemble`.

*   **`n_estimators=100`**: Specifies the number of trees in the forest (100 in this case).
*   **`random_state=42`**: Ensures reproducibility of the results.
*   The model is then fitted to the training data (`X_train`, `y_train`). Random Forest models learn by building multiple decision trees and merging their predictions.

A confirmation message is printed once the model is successfully trained.

In [11]:
y_pred = rf_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

RMSE : 347.52
MAE  : 232.16
R²   : 0.5989


## 9. Evaluating Random Forest Regressor Model (Onion Prices)

This cell evaluates the performance of the trained Random Forest Regressor model on the test set for 'Onion' prices using the same regression metrics as before:

1.  **Predictions**: The `rf_model.predict(X_test)` method generates price predictions for the test features.
2.  **Root Mean Squared Error (RMSE)**: Measures the average magnitude of the errors.
3.  **Mean Absolute Error (MAE)**: Measures the average absolute difference between predicted and actual values.
4.  **R-squared (R²)**: Indicates the proportion of the variance in the dependent variable explained by the model.

These metrics allow for a direct comparison of the Random Forest model's performance against the Linear Regression model for 'Onion' price prediction.

In [15]:
crop_name = 'Potato'    #then Tomato

crop_df = df[df['Commodity'] == crop_name].copy()

crop_df = crop_df.groupby('Price Date')['Modal_Price'].mean().reset_index()
crop_df = crop_df.sort_values('Price Date')

prices = crop_df['Modal_Price'].values

X = []
y = []

for i in range(LOOKBACK, len(prices)):
    X.append(prices[i-LOOKBACK:i])
    y.append(prices[i])

X = np.array(X)
y = np.array(y)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (527, 30)
Test : (132, 30)


## 10. Data Preparation and Train-Test Split for Potato Prices

This section repeats the data preparation and splitting process, but this time for 'Potato' prices:

1.  **`crop_name = 'Potato'`**: Sets the target commodity to 'Potato'.
2.  **Filter for 'Potato'**: The DataFrame is filtered to include only 'Potato' entries.
3.  **Group and Aggregate**: Daily `Modal_Price` is averaged for 'Potato'.
4.  **Sort by Date**: The 'Potato' data is sorted chronologically.
5.  **Feature Engineering**: `X` (lagged features) and `y` (target) arrays are created using the defined `LOOKBACK` period, similar to the 'Onion' data.
6.  **Train-Test Split**: The 'Potato' data is split into training and testing sets, maintaining chronological order.

The shapes of the training and testing sets are printed to confirm the successful preparation of the 'Potato' dataset for model training.

Now that we have prepared the data and trained both Linear Regression and Random Forest models for 'Potato', let's compare their performance metrics (RMSE, MAE, R²) to determine which model is more suitable for this crop.

### Evaluating the Linear Regression Model

Now, let's evaluate the performance of our Linear Regression model on the test data. We'll use three common regression metrics:

*   **Root Mean Squared Error (RMSE)**: This measures the average magnitude of the errors. It's the square root of the average of the squared differences between prediction and actual observation. Lower RMSE indicates a better fit.
*   **Mean Absolute Error (MAE)**: This measures the average magnitude of the errors, but without considering their direction. It's the average of the absolute differences between prediction and actual observation. MAE is less sensitive to outliers than RMSE.
*   **R-squared (R²)**: This represents the proportion of the variance in the dependent variable that is predictable from the independent variables. An R² of 1 indicates that the model explains all the variability of the response data around its mean, while an R² of 0 indicates that the model explains none of the variability.

In [16]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
lr_mae = mean_absolute_error(y_test, y_pred)
lr_r2 = r2_score(y_test, y_pred)

print("Linear Regression")
print(f"RMSE : {lr_rmse:.2f}")
print(f"MAE  : {lr_mae:.2f}")
print(f"R²   : {lr_r2:.4f}")

Linear Regression
RMSE : 347.65
MAE  : 243.40
R²   : 0.0180


### Evaluating the Random Forest Regressor Model

Next, we'll evaluate the Random Forest Regressor model using the same metrics: RMSE, MAE, and R². This will allow us to compare its performance directly with the Linear Regression model and determine which one provides a more accurate prediction for 'Potato' prices.

In [17]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rf_mae = mean_absolute_error(y_test, y_pred)
rf_r2 = r2_score(y_test, y_pred)

print("Random Forest")
print(f"RMSE : {rf_rmse:.2f}")
print(f"MAE  : {rf_mae:.2f}")
print(f"R²   : {rf_r2:.4f}")

Random Forest
RMSE : 335.80
MAE  : 240.93
R²   : 0.0838
